# Main Notebook
This notebook is only focused on getting text data, processing and sending notifications to the users, either by use `sms` or `email`.

In [ ]:
import pandas as pd
import numpy as np 
import re
import xmltodict as xd
import string
from dateetime import datetime 

In [ ]:
""" 
CREATE TABLE IF NOT EXISTS mpesa_messages (
    id INT AUTO_INCREMENT PRIMARY KEY,
    protocol INT,
    address VARCHAR(255),
    not_date INT,
    type INT,
    subject FLOAT,
    body TEXT,
    toa FLOAT,
    sc_toa FLOAT,
    service_center INT,
    read INT,
    status INT,
    locked INT,
    not_date_sent INT,
    sub_id INT,
    date DATE,
    contact_name VARCHAR(255)
);
"""


""" 
CREATE TABLE IF NOT EXISTS mpesa_transactions (
    id INT AUTO_INCREMENT PRIMARY KEY,
    unique_code VARCHAR(12),
    transaction_status VARCHAR(100),
    transaction_currency VARCHAR(3),
    transaction_amount FLOAT,
    transaction_date DATE,
    transaction_time TIME,
    float_balance FLOAT,
    transaction_type VARCHAR(100),
    message_id INT,
    FOREIGN KEY (message_id) REFERENCES mpesa_messages(id)
);
"""

""" 
CREATE TABLE IF NOT EXISTS customers(
    id INT AUTO_INCREMENT PRIMARY KEY,
    first_name VARCHAR(100),
    last_name VARCHAR(100)
);
"""



""" 
CREATE TABLE IF NOT EXISTS failed_transactions (
    id INT AUTO_INCREMENT PRIMARY KEY,
    message_id INT,
    date DATE,
    FOREIGN KEY (message_id) REFERENCES mpesa_messages(id)
);
"""

""" 
CREATE TABLE IF NOT EXISTS deposits (
    id INT AUTO_INCREMENT PRIMARY KEY,
    transaction_id INT,
    customer_id INT,
    amount FLOAT,
    date DATE,
    FOREIGN KEY (transaction_id) REFERENCES mpesa_transactions(id),
    FOREIGN KEY (customer_id) REFERENCES customers(id)
);

"""

""" 
CREATE TABLE IF NOT EXISTS withdrawals (
    id INT AUTO_INCREMENT PRIMARY KEY,
    transaction_id INT,
    customer_id INT,
    amount FLOAT,
    date DATE,
    FOREIGN KEY (transaction_id) REFERENCES mpesa_transactions(id),
    FOREIGN KEY (customer_id) REFERENCES customers(id));
"""

""" 
CREATE TABLE IF NOT EXISTS customer_transactions (
    id INT AUTO_INCREMENT PRIMARY KEY,
    customer_id INT,
    transaction_id INT,
    date DATE,
    FOREIGN KEY (customer_id) REFERENCES customers(id),
    FOREIGN KEY (transaction_id) REFERENCES mpesa_transactions(id));
"""

""" 
CREATE TABLE IF NOT EXISTS float_purchases (
    id INT AUTO_INCREMENT PRIMARY KEY,
    message_id INT,
    amount FLOAT,
    date DATE,
    FOREIGN KEY (message_id) REFERENCES mpesa_transactions(id)
);
"""

"""
CREATE TABLE IF NOT EXISTS deposit_commissions (
    id INT AUTO_INCREMENT PRIMARY KEY,
    range_min DECIMAL(10,2),
    range_max DECIMAL(10,2),
    agent_comm_reg DECIMAL(10,2),
    agent_comm_unreg DECIMAL(10,2),
    sub_agent_comm_reg DECIMAL(10,2),
    sub_agent_comm_unreg DECIMAL(10,2)
);
"""

"""
CREATE TABLE IF NOT EXISTS withdrawal_commissions (
    id INT AUTO_INCREMENT PRIMARY KEY,
    range_min DECIMAL(10,2),
    range_max DECIMAL(10,2),
    agent_comm_reg DECIMAL(10,2),
    agent_comm_unreg DECIMAL(10,2),
    sub_agent_comm_reg DECIMAL(10,2),
    sub_agent_comm_unreg DECIMAL(10,2)
);
"""

""" 
CREATE TABLE IF NOT EXISTS airtime_purchases (
    id INT AUTO_INCREMENT PRIMARY KEY,
    message_id INT,
    amount FLOAT,
    date DATE,
    FOREIGN KEY (message_id) REFERENCES mpesa_transactions(id)
);
"""

# COMMISSIONS_EARNED (if you want a single table to track both deposit & withdrawal commissions)
"""
CREATE TABLE IF NOT EXISTS commissions_earned (
    id INT AUTO_INCREMENT PRIMARY KEY,
    transaction_id INT NOT NULL,
    deposit_commission_id INT NULL,
    withdrawal_commission_id INT NULL,
    airtime_commission, DECIMAL(10,2),
    date DATE,
    FOREIGN KEY (transaction_id) REFERENCES mpesa_transactions(id),
    FOREIGN KEY (deposit_commission_id) REFERENCES deposit_commissions(id),
    FOREIGN KEY (withdrawal_commission_id) REFERENCES withdrawal_commissions(id),
    FOREIGN KEY (airtime_commission) REFERENCES airtime_commissions(id)
);
"""



#########
""" 
tables to create:
1. mpesa_messages ## done
2. mpesa_transactions ## done
3. failed_transactions ## done
4. deposits ## done
5. withdrawals ## done
6. float_purchases ## done
7. commissions ## done
8. commissions_earned ## done
9. customers ## done
10. customer_transactions ## done
11. airtime_purchases
"""

![Relationship Diagram](ERD_diagram.jpg)

In [4]:

import graphviz

# Create a new directed graph with left-to-right orientation
dot = graphviz.Digraph('ERD', format='jpg')
dot.attr(rankdir='LR', style='filled', color='lightgrey')

# Define nodes for each table with primary keys
dot.node('M', 'mpesa_messages\nid (PK)')
dot.node('T', 'mpesa_transactions\nid (PK)')
dot.node('F', 'failed_transactions\nid (PK)')
dot.node('C', 'customers\nid (PK)')
dot.node('D', 'deposits\nid (PK)')
dot.node('W', 'withdrawals\nid (PK)')
dot.node('CT', 'customer_transactions\nid (PK)')
dot.node('FP', 'float_purchases\nid (PK)')
dot.node('AP', 'airtime_purchases\nid (PK)')
dot.node('DC', 'deposit_commissions\nid (PK)')
dot.node('WC', 'withdrawal_commissions\nid (PK)')
dot.node('AC', 'airtime_commissions\nid (PK)')
dot.node('CE', 'commissions_earned\nid (PK)')

# Define edges (foreign key relationships)
dot.edge('M', 'T', label="FK message_id")
dot.edge('M', 'F', label="FK message_id")
dot.edge('T', 'D', label="FK transaction_id")
dot.edge('T', 'W', label="FK transaction_id")
dot.edge('T', 'FP', label="FK transaction_id")
dot.edge('T', 'AP', label="FK transaction_id")
dot.edge('C', 'D', label="FK customer_id")
dot.edge('C', 'W', label="FK customer_id")
dot.edge('C', 'CT', label="FK customer_id")
dot.edge('T', 'CT', label="FK transaction_id")
dot.edge('DC', 'CE', label="FK deposit_commission_id")
dot.edge('WC', 'CE', label="FK withdrawal_commission_id")
dot.edge('AC', 'CE', label="FK airtime_commission")
dot.edge('T', 'CE', label="FK transaction_id")

# Save and render the updated ERD diagram as a JPG file
output_path = dot.render('ERD_diagram', cleanup=True)
print("ERD diagram generated and saved as:", output_path)


ERD diagram generated and saved as: ERD_diagram.jpg


In [ ]:
###pick up from here tomorrow, analyse the erd to see if it makes sense and adjust the tables accordingly